# 12 Merge Inflected Forms

The lemmatizer (step 05) leaves some inflected forms as their own word, mostly `-ing` words it reads as nouns: `gambling` stays `gambling` while `gamble` is a separate feature, so one idea is split across two rows. This step finds those pairs in the final words and **merges the inflected form into its base (dictionary) form**, for example `feeling → feel`, `gambling → gamble`.

A pair is merged only when all of these hold:

1. the word is the base plus an inflection (`-ing`, `-ed`, `-s`, `-es`, `-ies`, `-ied`, with the usual spelling changes) and the base is also a word in the data,
2. WordNet agrees that the base is the root of the word (this blocks `united → unit`, `scared → scar`, `caring → car`, `news → new`),
3. the base is not much rarer than the word (at least half as common). The base does not have to be the most common form (`gambling` is more common than `gamble` here, but `gamble` is the lemma), but a far rarer base is usually a different word (`wedding`/`wed`, `annoying`/`annoy`, `clothes`/`clothe`),
4. the word is not on the keep-separate list `dictionaries/lemma_keep_separate.csv`, which holds words whose meaning changes (`glasses`, `states`, `evening`, ...). Edit that file and rerun to change the result.

Derivational forms (`good`/`goodness`, `person`/`personal`) are never merged.

Outputs: `dictionaries/lemma_merge.csv` (the rule list; the LLM pipeline reads the same file) and `processed/12_cue_feature_counts_merged.csv`, laid out like the 2019 norms: one row per original cue-word pair (`feature`, `frequency_feature`) with the merged word next to it (`lemma`, `frequency_lemma`). For example `gambling | gamble | 8 | 12` means 8 people wrote gambling and 12 wrote gambling or gamble (a person who wrote both counts once).

| Column | Meaning |
|---|---|
| `cue` | the cue word |
| `feature` | the word as it came out of step 08 |
| `lemma` | the word it is merged into (the same word if nothing is merged) |
| `frequency_feature` | people who listed `feature` (same as step 09 `count`) |
| `frequency_lemma` | people who listed `lemma` or anything merged into it |
| `n` | people with at least one word for this cue (`n_final`) |
| `normalized_feature`, `normalized_lemma` | the two frequencies as a percent of `n` |
| `n_answered`, `pos`, `mean_position`, `original_words` | as in step 09 |

In [1]:
import pandas as pd
from nltk.corpus import wordnet as wn

In [2]:
words = pd.read_csv("processed/08_words.csv", dtype={"subject_id": str, "position": int,
                                                       "n_answered": int, "n_final": int}, keep_default_na=False)
n_word = words["word"].value_counts().to_dict()
keep_separate = set(pd.read_csv("dictionaries/lemma_keep_separate.csv")["word"])
print(f"{len(words):,} rows, {len(n_word):,} distinct words, {len(keep_separate)} on the keep-separate list")

952,792 rows, 29,913 distinct words, 17 on the keep-separate list


In [3]:
RULES = [("ing", ""), ("ing", "e"), ("ed", ""), ("ed", "e"), ("ies", "y"), ("ied", "y"), ("es", ""), ("s", "")]

def candidate_bases(w):
    out = []
    for suffix, replacement in RULES:
        if w.endswith(suffix) and len(w) - len(suffix) >= 3:
            b = w[: -len(suffix)] + replacement
            out.append(b)
            if suffix in ("ing", "ed") and b[-1] == b[-2]:   # swimming -> swimm -> swim
                out.append(b[:-1])
    return out

def wordnet_root(w, b):
    return any(wn.morphy(w, pos) == b for pos in ("v", "n", "a"))

rows = []
for w, n in n_word.items():
    if w in keep_separate:
        continue
    bases = [b for b in candidate_bases(w) if b != w and 2 * n_word.get(b, 0) >= n and wordnet_root(w, b)]
    if bases:
        base = max(bases, key=lambda b: n_word[b])
        rows.append((w, base, n, n_word[base]))
merge = pd.DataFrame(rows, columns=["word", "merge_to", "n_word", "n_merge_to"]).sort_values("n_word", ascending=False)
merge.to_csv("dictionaries/lemma_merge.csv", index=False)
print(f"{len(merge):,} words merged, covering {merge['n_word'].sum():,} rows ({merge['n_word'].sum() / len(words):.1%})")
merge.head(40)

2,576 words merged, covering 21,173 rows (2.2%)


,word,merge_to,n_word,n_merge_to
0,feeling,feel,1406,1684
1,living,live,384,1870
2,cooking,cook,293,645
3,writing,write,271,918
4,beginning,begin,245,205
5,swimming,swim,215,218
6,meeting,meet,214,235
7,tired,tire,209,126
8,interesting,interest,179,201
9,shopping,shop,173,204


## Apply the merge and recount

In [4]:
mapping = dict(zip(merge["word"], merge["merge_to"]))
old = pd.read_csv("processed/09_cue_feature_counts.csv", keep_default_na=False)   # original counts

# people per (cue, merged word); a person who wrote both forms counts once
words["lemma"] = words["word"].map(lambda w: mapping.get(w, w))
merged = (words.groupby(["cue", "lemma"])["subject_id"].nunique()
               .rename("frequency_lemma").reset_index())

out = old.rename(columns={"word": "feature", "count": "frequency_feature", "n_final": "n"})
out["lemma"] = out["feature"].map(lambda w: mapping.get(w, w))
out = out.merge(merged, on=["cue", "lemma"], how="left")
out["normalized_feature"] = (out["frequency_feature"] / out["n"] * 100).round(4)
out["normalized_lemma"] = (out["frequency_lemma"] / out["n"] * 100).round(4)
out = out[["cue", "feature", "lemma", "frequency_feature", "frequency_lemma", "n",
           "normalized_feature", "normalized_lemma", "n_answered", "pos", "mean_position", "original_words"]]
out = out.sort_values(["cue", "frequency_lemma", "lemma", "frequency_feature"], ascending=[True, False, True, False])
assert (out["frequency_lemma"] >= out["frequency_feature"]).all()
assert (out["frequency_lemma"] <= out["n"]).all()
print(f"{len(out):,} cue-word rows; {(out['feature'] != out['lemma']).sum():,} are merged into another word")

429,486 cue-word rows; 13,515 are merged into another word


## Check: how much did the top features change?

In [5]:
merged_pairs = out.drop_duplicates(["cue", "lemma"])
def top10(df, col, word):
    return df.sort_values([col, word], ascending=[False, True]).groupby("cue").head(10)
o = top10(out, "frequency_feature", "feature")
m = top10(merged_pairs, "frequency_lemma", "lemma")
changed = set(zip(m["cue"], m["lemma"])) - set(zip(o["cue"], o["feature"]))
print(f"{len(changed):,} of {len(m):,} top-10 cue-word slots differ after merging")
out[out["cue"] == "chance"].head(12)

1,083 of 37,220 top-10 cue-word slots differ after merging


,cue,feature,lemma,frequency_feature,frequency_lemma,n,normalized_feature,normalized_lemma,n_answered,pos,mean_position,original_words
62960,chance,opportunity,opportunity,17,17,68,25.0000,25.0000,70,NOUN,1.41,opportunity
62961,chance,luck,luck,13,13,68,19.1176,19.1176,70,NOUN,2.46,luck
62963,chance,gambling,gamble,8,12,68,11.7647,17.6471,70,NOUN,3.62,gambling
62972,chance,gamble,gamble,4,12,68,5.8824,17.6471,70,NOUN,2.25,gamble
62962,chance,risk,risk,12,12,68,17.6471,17.6471,70,NOUN,2.33,risk|risks
62964,chance,happen,happen,8,8,68,11.7647,11.7647,70,VERB,3.38,happen|happening|happens
62965,chance,probability,probability,8,8,68,11.7647,11.7647,70,NOUN,2.88,probability
62966,chance,game,game,7,7,68,10.2941,10.2941,70,NOUN,4.00,game|games
62967,chance,monopoly,monopoly,6,6,68,8.8235,8.8235,70,NOUN,3.67,monopoly
62968,chance,name,name,6,6,68,8.8235,8.8235,70,NOUN,3.50,name


In [6]:
out.to_csv("processed/12_cue_feature_counts_merged.csv", index=False)
print(f"Saved {len(out):,} rows to processed/12_cue_feature_counts_merged.csv")

Saved 429,486 rows to processed/12_cue_feature_counts_merged.csv
